In [40]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [41]:
torch.manual_seed(42)

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [43]:
df = pd.read_csv('fashion-mnist_train.csv')

In [44]:
# Clean NaN values from the dataset
print(f"Initial rows with NaNs: {df.isna().any(axis=1).sum()}")
df = df.dropna()
print(f"Rows remaining after dropping NaNs: {len(df)}")

Initial rows with NaNs: 0
Rows remaining after dropping NaNs: 60000


In [45]:
# Re-extract features and labels after dropping NaNs
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [46]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2, random_state = 42)

In [47]:
#transformations
from torchvision.transforms import transforms

custom_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229,0.224,0.225])
])

In [48]:
from PIL import Image
import numpy as np

class CustomDataset(Dataset):

  def __init__(self, features, labels, transforms):
    self.features = features
    self.labels = labels
    self.transforms = transforms

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    #resize to (28,28)
    image = self.features[index].reshape(28,28)

    #ensure datatype is np.uint8 for PIL
    image = image.astype(np.uint8)

    #change b&w to 3-channel RGB -> (H,W,C)
    image = np.stack([image]*3, axis = -1)

    #convert array to PIL image
    image = Image.fromarray(image)

    #apply transforms
    image = self.transforms(image)

    #return image and label
    return image, torch.tensor(int(self.labels[index]), dtype = torch.long)


In [49]:
#create train_dataset object

train_dataset = CustomDataset(X_train, y_train, transforms=custom_transforms)

In [50]:
#create test_dataset object
test_dataset = CustomDataset(X_test, y_test, transforms=custom_transforms)

In [51]:
#create train and test loader
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True, pin_memory = True)